In [1]:
# Optional speedup cell: transparently prefer Parquet for tracking_week_*.csv reads.
# Run this once before Cells 2-4.
from pathlib import Path
import re
import pandas as pd

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

if not hasattr(pd, "_orig_read_csv"):
    pd._orig_read_csv = pd.read_csv


def _resolve_usecols_for_parquet(parquet_path: Path, usecols):
    if usecols is None:
        return None
    if callable(usecols):
        if pq is None:
            return None
        cols = pq.ParquetFile(parquet_path).schema.names
        return [c for c in cols if usecols(c)]
    if isinstance(usecols, (list, tuple, set)):
        return list(usecols)
    return None

def _read_csv_preferring_parquet(path, *args, **kwargs):
    p = Path(path) if isinstance(path, (str, Path)) else None
    if p is not None and re.search(r"tracking_week_\d+\.csv$", p.name):
        parquet_path = p.with_suffix(".parquet")
        if parquet_path.exists():
            usecols = kwargs.pop("usecols", None)
            cols = _resolve_usecols_for_parquet(parquet_path, usecols)
            out = pd.read_parquet(parquet_path, columns=cols)
            return out
    return pd._orig_read_csv(path, *args, **kwargs)

pd.read_csv = _read_csv_preferring_parquet
print("Parquet preference enabled for tracking_week_*.csv reads.")

Parquet preference enabled for tracking_week_*.csv reads.


In [2]:
#Run/Pass Probability Model (tracking plays only)
# Features: down, yardsToGo, quarter, absoluteYardlineNumber,
#           offenseFormation, receiverAlignment, scoreDiff, gameClockSeconds
# Target  : 1 = pass, 0 = run
# All features are purely PRE-SNAP 
# isDropback is used only to LABEL plays, not as a model input.

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

#1. Load data
plays = pd.read_csv(Path(".") / "plays.csv")

#2. Label plays (isDropback used HERE only â€” not as a model input)
def label_target(df):
    s = pd.Series(np.nan, index=df.index)
    if "passResult" in df.columns:
        pr = df["passResult"].astype(str).str.upper().str.strip()
        s[pr.isin(["C", "I", "IN", "S"])] = 1
        s[pr.eq("R")] = 0
    if "isDropback" in df.columns:
        db = df["isDropback"].astype(str).str.upper().str.strip()
        s[(db == "TRUE") | (db == "1")] = 1
    if "rushLocationType" in df.columns:
        s[df["rushLocationType"].notna()] = 0
    return s

plays["target"] = label_target(plays)

# Drop QB kneel/spike, not real play calls
labeled = plays[plays["target"].isin([0, 1])].copy()
labeled = labeled[~labeled["qbSpike"].astype(str).str.upper().eq("TRUE")]
labeled = labeled[~(labeled["qbKneel"] == 1)]

# 3. Pre-snap feature engineering
def clock_to_seconds(val):
    """'MM:SS' â†’ total seconds remaining."""
    try:
        m, s = str(val).split(":")
        return int(m) * 60 + int(s)
    except Exception:
        return np.nan

labeled["gameClockSeconds"] = labeled["gameClock"].apply(clock_to_seconds)

# Score differential from the offensive team's perspective
home_mask = labeled["possessionTeam"] == labeled.get("homeTeam", pd.Series("", index=labeled.index))
labeled["scoreDiff"] = np.where(
    labeled["possessionTeam"] == labeled.get("homeTeam", pd.Series("", index=labeled.index)),
    labeled["preSnapHomeScore"] - labeled["preSnapVisitorScore"],
    labeled["preSnapVisitorScore"] - labeled["preSnapHomeScore"],
)
# Fallback: if possessionTeam/homeTeam not comparable, use raw diff (still informative)
labeled["scoreDiff"] = labeled["scoreDiff"].fillna(
    labeled["preSnapHomeScore"] - labeled["preSnapVisitorScore"]
)

NUM_FEATS = ["down", "yardsToGo", "quarter", "absoluteYardlineNumber",
             "gameClockSeconds", "scoreDiff"]
CAT_FEATS = ["offenseFormation", "receiverAlignment"]

X = labeled[NUM_FEATS + CAT_FEATS].copy()
y = labeled["target"].astype(int)

# 4. Preprocessing + model pipeline
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("ohe",    OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, NUM_FEATS),
    ("cat", cat_pipe, CAT_FEATS),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf",  GradientBoostingClassifier(
                 n_estimators=400,
                 learning_rate=0.05,
                 max_depth=4,
                 subsample=0.8,
                 random_state=42)),
])

# 5. Train / evaluate 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

proba_test = model.predict_proba(X_test)[:, 1]
pred_test  = model.predict(X_test)

# 6. Feature importances
feat_names = (
    NUM_FEATS
    + list(model.named_steps["prep"]
               .named_transformers_["cat"]
               .named_steps["ohe"]
               .get_feature_names_out(CAT_FEATS))
)
importances = pd.Series(
    model.named_steps["clf"].feature_importances_,
    index=feat_names
).sort_values(ascending=False)

print("\n Top feature importances")
print(importances.head(10).to_string())

# 7. Pre-snap prediction helper 
def predict_pass_prob(
    down: int,
    yards_to_go: int,
    quarter: int,
    absolute_yardline: int,
    offense_formation: str = "SHOTGUN",
    receiver_alignment: str = "2x2",
    game_clock_seconds: float = np.nan,
    score_diff: float = 0.0,
) -> dict:
    """
    Returns predicted pass probability using ONLY pre-snap situation features.

    Parameters
    ----------
    down                : 1â€“4
    yards_to_go         : yards needed for first down
    quarter             : 1â€“5  (5 = OT)
    absolute_yardline   : yards from offensive team's own end zone (1â€“99)
    offense_formation   : "SHOTGUN", "SINGLEBACK", "I_FORM", "EMPTY", etc.
    receiver_alignment  : "2x2", "3x1", "2x1", etc.
    game_clock_seconds  : seconds remaining on game clock (e.g. 420 = 7:00)
    score_diff          : offensive team's score minus defensive team's score
    """
    row = pd.DataFrame([{
        "down":                  down,
        "yardsToGo":             yards_to_go,
        "quarter":               quarter,
        "absoluteYardlineNumber": absolute_yardline,
        "gameClockSeconds":      game_clock_seconds,
        "scoreDiff":             score_diff,
        "offenseFormation":      offense_formation.upper().strip(),
        "receiverAlignment":     receiver_alignment.upper().strip(),
    }])
    prob_pass = model.predict_proba(row)[0, 1]
    return {
        "pass_probability": round(prob_pass, 4),
        "run_probability":  round(1 - prob_pass, 4),
        "prediction":       "PASS" if prob_pass >= 0.5 else "RUN",
    }

print("\n Example predictions (pre-snap only)")
examples = [
    dict(down=3, yards_to_go=7,  quarter=3, absolute_yardline=65, offense_formation="SHOTGUN",    score_diff=-3),
    dict(down=1, yards_to_go=10, quarter=1, absolute_yardline=75, offense_formation="SINGLEBACK", score_diff=0),
    dict(down=4, yards_to_go=1,  quarter=4, absolute_yardline=55, offense_formation="I_FORM",     score_diff=7),
    dict(down=2, yards_to_go=8,  quarter=2, absolute_yardline=50, offense_formation="EMPTY",      score_diff=-10),
]
for ex in examples:
    result = predict_pass_prob(**ex)
    label  = f"D{ex['down']}&{ex['yards_to_go']:>2} Q{ex['quarter']} {ex['offense_formation']:<12} diff={ex['score_diff']:+d}"
    print(f"  {label}, {result['prediction']}  (pass {result['pass_probability']:.1%})")



 Top feature importances
offenseFormation_SINGLEBACK    0.206629
gameClockSeconds               0.128952
yardsToGo                      0.099195
receiverAlignment_2x1          0.095621
offenseFormation_SHOTGUN       0.094592
offenseFormation_EMPTY         0.078645
absoluteYardlineNumber         0.072995
down                           0.067841
scoreDiff                      0.056734
quarter                        0.031406

 Example predictions (pre-snap only)
  D3& 7 Q3 SHOTGUN      diff=-3, PASS  (pass 87.0%)
  D1&10 Q1 SINGLEBACK   diff=+0, RUN  (pass 27.3%)
  D4& 1 Q4 I_FORM       diff=+7, RUN  (pass 18.6%)
  D2& 8 Q2 EMPTY        diff=-10, PASS  (pass 95.1%)


In [ ]:
# Train and classify using every available local tracking week/play.
# The model uses all labeled tracking plays from plays.csv, then scores the same tracking universe.
# If one of the five offense players aligned closest to the line of scrimmage gets >1 yard downfield,
# force the play classification to run.

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display
from tracking_data_io import resolve_tracking_files, read_tracking_file, week_num_from_name as tracking_week_num_from_name

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_DIR = Path(".")
SEASONS = [2022, 2023, 2024, 2025]

# Use every local tracking week.
SELECTED_WEEKS = None

plays_path = DATA_DIR / "plays.csv"
tracking_candidates = resolve_tracking_files(DATA_DIR)

if not plays_path.exists() or not tracking_candidates:
    raise FileNotFoundError("Need plays.csv and tracking_week_*.parquet (or tracking_week_*.csv fallback) in the notebook folder.")

def week_num_from_name(path_obj):
    return tracking_week_num_from_name(path_obj)

def label_pass_run(df):
    out = df.copy()
    out["is_pass"] = np.nan

    if "passResult" in out.columns:
        pr = out["passResult"].astype(str).str.upper().fillna("")
        out.loc[pr.isin(["C", "I", "IN", "S"]), "is_pass"] = 1
        out.loc[pr.eq("R"), "is_pass"] = 0

    if "isDropback" in out.columns:
        out.loc[out["isDropback"] == 1, "is_pass"] = 1

    if "rushAttempt" in out.columns:
        out.loc[out["rushAttempt"] == 1, "is_pass"] = 0
    elif "rushLocationType" in out.columns:
        out.loc[out["rushLocationType"].notna(), "is_pass"] = 0

    out = out.loc[out["is_pass"].isin([0, 1])].copy()
    out["is_pass"] = out["is_pass"].astype(int)
    return out

def ordinal_down(value):
    mapping = {1: "1st", 2: "2nd", 3: "3rd", 4: "4th"}
    if pd.isna(value):
        return ""
    return mapping.get(int(value), f"{int(value)}th")

def build_situation_text(row):
    parts = []
    if pd.notna(row.get("down")) and pd.notna(row.get("yards_to_go")):
        parts.append(f"{ordinal_down(row['down'])} & {int(row['yards_to_go'])}")

    quarter = row.get("quarter")
    game_clock = row.get("game_clock")
    if pd.notna(quarter) or pd.notna(game_clock):
        quarter_text = f"Q{int(quarter)}" if pd.notna(quarter) else ""
        clock_text = str(game_clock) if pd.notna(game_clock) else ""
        parts.append(" ".join(p for p in [quarter_text, clock_text] if p))

    yardline_side = row.get("yardlineSide")
    yardline_number = row.get("yardlineNumber")
    if pd.notna(yardline_side) and pd.notna(yardline_number):
        parts.append(f"{yardline_side} {int(yardline_number)}")

    return " | ".join(part for part in parts if part)


def build_downfield_run_rule(tracking_df, plays_df):
    if tracking_df.empty:
        return pd.DataFrame(columns=["gameId", "playId", "linemen_downfield_run_rule", "max_ol_downfield", "line_of_scrimmage"])
    tracking_std = tracking_df.copy()
    tracking_std["playDirection"] = tracking_std["playDirection"].astype(str).str.lower()
    tracking_std["x_std"] = np.where(tracking_std["playDirection"].eq("left"), 120 - tracking_std["x"], tracking_std["x"])
    tracking_std["y_std"] = np.where(tracking_std["playDirection"].eq("left"), 53.3 - tracking_std["y"], tracking_std["y"])
    tracking_std = tracking_std.merge(
        plays_df[["gameId", "playId", "possessionTeam"]].drop_duplicates(),
        on=["gameId", "playId"],
        how="left",
    )

    snap_frames = (
        tracking_std.loc[tracking_std["event"].eq("ball_snap")]
        .groupby(["gameId", "playId"], as_index=False)["frameId"]
        .min()
        .rename(columns={"frameId": "snap_frame"})
    )
    tracking_std = tracking_std.merge(snap_frames, on=["gameId", "playId"], how="left")
    before_snap = tracking_std.loc[tracking_std["frameType"].eq("BEFORE_SNAP")].copy()
    before_snap = before_snap.loc[
        before_snap["snap_frame"].isna() | (before_snap["frameId"] <= before_snap["snap_frame"])
    ]

    pre_snap_frame = (
        before_snap.groupby(["gameId", "playId"], as_index=False)["frameId"]
        .max()
        .rename(columns={"frameId": "pre_snap_frame"})
    )
    pre_snap = before_snap.merge(pre_snap_frame, on=["gameId", "playId"], how="inner")
    pre_snap = pre_snap.loc[pre_snap["frameId"] == pre_snap["pre_snap_frame"]].copy()

    ball_los = pre_snap.loc[pre_snap["club"].eq("football"), ["gameId", "playId", "x_std"]].rename(
        columns={"x_std": "line_of_scrimmage"}
    )

    offense_pre_snap = pre_snap.loc[pre_snap["club"] == pre_snap["possessionTeam"]].copy()
    offense_pre_snap = offense_pre_snap.merge(ball_los, on=["gameId", "playId"], how="left")
    offense_pre_snap["distance_to_los"] = (offense_pre_snap["x_std"] - offense_pre_snap["line_of_scrimmage"]).abs()

    line_proxy = (
        offense_pre_snap.sort_values(["gameId", "playId", "distance_to_los", "y_std", "nflId"])
        .groupby(["gameId", "playId"], group_keys=False)
        .head(5)
        [["gameId", "playId", "nflId"]]
        .drop_duplicates()
    )

    post_snap = tracking_std.loc[
        tracking_std["snap_frame"].notna() & (tracking_std["frameId"] > tracking_std["snap_frame"])
    ].copy()
    post_snap = post_snap.loc[post_snap["club"] == post_snap["possessionTeam"]].copy()
    post_snap = post_snap.merge(line_proxy, on=["gameId", "playId", "nflId"], how="inner")
    post_snap = post_snap.merge(ball_los, on=["gameId", "playId"], how="left")
    post_snap["yards_downfield"] = post_snap["x_std"] - post_snap["line_of_scrimmage"]

    run_rule = (
        post_snap.groupby(["gameId", "playId"], as_index=False)
        .agg(
            max_ol_downfield=("yards_downfield", "max"),
            line_of_scrimmage=("line_of_scrimmage", "first"),
        )
    )
    run_rule["max_ol_downfield"] = run_rule["max_ol_downfield"].fillna(0)
    run_rule["linemen_downfield_run_rule"] = (run_rule["max_ol_downfield"] > 1.0).astype(int)
    return run_rule


if SELECTED_WEEKS is None:
    tracking_files = tracking_candidates
else:
    wanted = set(int(w) for w in SELECTED_WEEKS)
    tracking_files = [p for p in tracking_candidates if week_num_from_name(p) in wanted]

if not tracking_files:
    raise ValueError(f"No tracking files matched SELECTED_WEEKS={SELECTED_WEEKS}.")

selected_weeks_resolved = sorted({week_num_from_name(p) for p in tracking_files if week_num_from_name(p) is not None})
print(f"plays.csv found: {plays_path.name}")
print(f"tracking weeks used for modeling: {selected_weeks_resolved}")
print(f"tracking files used: {[p.name for p in tracking_files]}")

plays = pd.read_csv(plays_path)
plays = label_pass_run(plays)
plays["season"] = (plays["gameId"] // 1000000).astype(int)
plays = plays.loc[plays["season"].isin(SEASONS)].copy()

plays_features = [
    "gameId", "playId", "season", "is_pass", "down", "yardsToGo", "quarter",
    "absoluteYardlineNumber", "offenseFormation", "receiverAlignment", "playAction",
    "pff_runPassOption", "preSnapHomeScore", "preSnapVisitorScore",
    "possessionTeam", "defensiveTeam", "gameClock", "playDescription",
    "yardlineSide", "yardlineNumber",
]
plays_features = [c for c in plays_features if c in plays.columns]
plays_model = plays[plays_features].copy()

rename_map = {
    "yardsToGo": "yards_to_go",
    "absoluteYardlineNumber": "yardline",
    "offenseFormation": "offense_formation",
    "receiverAlignment": "receiver_alignment",
    "playAction": "play_action",
    "pff_runPassOption": "run_pass_option",
    "gameClock": "game_clock",
    "playDescription": "play_description",
}
plays_model = plays_model.rename(columns={k: v for k, v in rename_map.items() if k in plays_model.columns})

if {"preSnapHomeScore", "preSnapVisitorScore"}.issubset(plays_model.columns):
    plays_model["score_total"] = plays_model["preSnapHomeScore"].fillna(0) + plays_model["preSnapVisitorScore"].fillna(0)

tracking_cols = ["gameId", "playId", "nflId", "frameId", "frameType", "club", "playDirection", "x", "y", "s", "event"]
tracking_parts = []
for path in tracking_files:
    part = read_tracking_file(path, columns=tracking_cols)
    part["tracking_week"] = week_num_from_name(path)
    tracking_parts.append(part)

tracking = pd.concat(tracking_parts, ignore_index=True)
tracking_keys = tracking[["gameId", "playId"]].drop_duplicates()
run_rule = build_downfield_run_rule(tracking, plays_model)

# Train on every labeled tracking play.
train_df = tracking_keys.merge(plays_model, on=["gameId", "playId"], how="left")
train_df = train_df.merge(run_rule, on=["gameId", "playId"], how="left")
train_df["linemen_downfield_run_rule"] = train_df["linemen_downfield_run_rule"].fillna(0).astype(int)
train_df["max_ol_downfield"] = train_df["max_ol_downfield"].fillna(0.0)
train_df.loc[train_df["linemen_downfield_run_rule"] == 1, "is_pass"] = 0

infer_df = tracking_keys.merge(plays_model, on=["gameId", "playId"], how="left")
infer_df = infer_df.merge(run_rule, on=["gameId", "playId"], how="left")
infer_df["linemen_downfield_run_rule"] = infer_df["linemen_downfield_run_rule"].fillna(0).astype(int)
infer_df["max_ol_downfield"] = infer_df["max_ol_downfield"].fillna(0.0)

required_train = [c for c in ["down", "yards_to_go", "yardline", "quarter"] if c in train_df.columns]
train_df = train_df.dropna(subset=required_train + ["is_pass"]).copy()
train_df["is_pass"] = train_df["is_pass"].astype(int)

feature_cols = [
    "season", "down", "yards_to_go", "quarter", "yardline",
    "offense_formation", "receiver_alignment", "play_action", "run_pass_option",
    "preSnapHomeScore", "preSnapVisitorScore", "score_total", "possessionTeam", "defensiveTeam",
    "linemen_downfield_run_rule", "max_ol_downfield",
]
feature_cols = [c for c in feature_cols if c in train_df.columns and c in infer_df.columns]

print(f"Training rows (all labeled tracking plays): {len(train_df):,}")
print("Training target balance after run-rule override:")
print(f"Tracking plays scored: {len(infer_df):,}")

if train_df["is_pass"].nunique() < 2:
    raise ValueError("Training data has only one class. Verify the local tracking/plays inputs.")

X_train_all = train_df[feature_cols].copy()
y_train_all = train_df["is_pass"].astype(int)
X_infer = infer_df[feature_cols].copy()

cat_cols = X_train_all.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

for c in cat_cols:
    X_train_all[c] = X_train_all[c].fillna("missing").astype(str)
    X_infer[c] = X_infer[c].fillna("missing").astype(str)

prep = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

clf = Pipeline(
    steps=[
        ("prep", prep),
        ("rf", RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42, n_jobs=-1)),
    ]
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_train_all,
    y_train_all,
    test_size=0.2,
    stratify=y_train_all,
    random_state=42,
)
clf.fit(X_tr, y_tr)
proba_te = clf.predict_proba(X_te)[:, 1]
pred_te = (proba_te >= 0.5).astype(int)

print(f"\nModel validation ROC AUC (tracking-play holdout): {roc_auc_score(y_te, proba_te):.3f}")
print(classification_report(y_te, pred_te, digits=3))

clf.fit(X_train_all, y_train_all)
infer_proba = clf.predict_proba(X_infer)[:, 1]
infer_pred = (infer_proba >= 0.5).astype(int)

predictions = infer_df[["gameId", "playId"]].copy()
predictions["pred_pass_prob"] = infer_proba
predictions["pred_label"] = np.where(infer_pred == 1, "pass", "run")
predictions["linemen_downfield_run_rule"] = infer_df["linemen_downfield_run_rule"].to_numpy(dtype=int)
predictions["max_ol_downfield"] = infer_df["max_ol_downfield"].to_numpy(dtype=float)

rule_mask = predictions["linemen_downfield_run_rule"].eq(1)
predictions.loc[rule_mask, "pred_pass_prob"] = np.minimum(predictions.loc[rule_mask, "pred_pass_prob"], 0.02)
predictions.loc[rule_mask, "pred_label"] = "run"

if "is_pass" in infer_df.columns:
    predictions["true_label_from_plays"] = np.where(infer_df["is_pass"] == 1, "pass", "run")

tracking = tracking.merge(predictions[["gameId", "playId"]], on=["gameId", "playId"], how="inner")
tracking["playDirection"] = tracking["playDirection"].astype(str).str.lower()
tracking["x_std"] = np.where(tracking["playDirection"].eq("left"), 120 - tracking["x"], tracking["x"])
tracking["y_std"] = np.where(tracking["playDirection"].eq("left"), 53.3 - tracking["y"], tracking["y"])

snap_frames = (
    tracking.loc[tracking["event"].eq("ball_snap")]
    .groupby(["gameId", "playId"], as_index=False)["frameId"]
    .min()
    .rename(columns={"frameId": "snap_frame"})
)
before_snap = tracking.loc[tracking["frameType"].eq("BEFORE_SNAP")].copy()
before_snap = before_snap.merge(snap_frames, on=["gameId", "playId"], how="left")
before_snap = before_snap.loc[before_snap["snap_frame"].isna() | (before_snap["frameId"] <= before_snap["snap_frame"])]

pre_snap_frame = (
    before_snap.groupby(["gameId", "playId"], as_index=False)["frameId"]
    .max()
    .rename(columns={"frameId": "pre_snap_frame"})
)
pre_snap = before_snap.merge(pre_snap_frame, on=["gameId", "playId"], how="inner")
pre_snap = pre_snap.loc[pre_snap["frameId"] == pre_snap["pre_snap_frame"]].copy()

ball_loc = pre_snap.loc[pre_snap["club"].eq("football"), ["gameId", "playId", "x_std", "y_std"]].rename(columns={"x_std": "ball_x", "y_std": "ball_y"})
pre_snap = pre_snap.merge(ball_loc, on=["gameId", "playId"], how="left")
pre_snap["dist_from_ball"] = np.hypot(pre_snap["x_std"] - pre_snap["ball_x"], pre_snap["y_std"] - pre_snap["ball_y"])

tracking_summary = pre_snap.groupby(["gameId", "playId"], as_index=False).agg(
    avg_dist_from_ball=("dist_from_ball", "mean"),
    max_player_speed_presnap=("s", "max"),
    player_count_presnap=("nflId", "nunique"),
)

context_cols = [
    "gameId", "playId", "down", "yards_to_go", "quarter", "game_clock",
    "yardlineSide", "yardlineNumber", "possessionTeam", "play_description",
]
context_cols = [c for c in context_cols if c in infer_df.columns]
play_context = infer_df[context_cols].copy().drop_duplicates(subset=["gameId", "playId"])
if {"down", "yards_to_go", "quarter", "game_clock", "yardlineSide", "yardlineNumber"}.intersection(play_context.columns):
    play_context["situation_text"] = play_context.apply(build_situation_text, axis=1)
else:
    play_context["situation_text"] = ""

final_tracking_classification = predictions.merge(tracking_summary, on=["gameId", "playId"], how="left")
final_tracking_classification = final_tracking_classification.merge(play_context, on=["gameId", "playId"], how="left")
final_tracking_classification = final_tracking_classification.sort_values("pred_pass_prob", ascending=False)

print("\nTracking play classifications:")
display(final_tracking_classification.head(3))

plays.csv found: plays.csv
tracking weeks used for modeling: []
tracking files used: ['tracking_all_weeks.parquet']
Training rows (all labeled tracking plays): 16,124
Training target balance after run-rule override:
Tracking plays scored: 16,124

Model validation ROC AUC (tracking-play holdout): 0.998
              precision    recall  f1-score   support

           0      0.997     1.000     0.998      3203
           1      1.000     0.500     0.667        22

    accuracy                          0.997      3225
   macro avg      0.998     0.750     0.832      3225
weighted avg      0.997     0.997     0.996      3225


Tracking play classifications:


,gameId,playId,pred_pass_prob,pred_label,linemen_downfield_run_rule,max_ol_downfield,true_label_from_plays,avg_dist_from_ball,max_player_speed_presnap,player_count_presnap,down,yards_to_go,quarter,game_clock,yardlineSide,yardlineNumber,possessionTeam,play_description,situation_text
14866,2022110609,1134,0.926965,pass,0,-0.230001,pass,6.170528,4.40,22.0,2,6,2,12:08,TB,29,TB,(12:08) T.Brady pass incomplete short middle.,2nd & 6 | Q2 12:08 | TB 29
15157,2022110607,2602,0.906578,pass,0,-0.660002,pass,5.434820,4.72,22.0,1,10,3,02:23,WAS,44,WAS,(2:23) T.Heinicke pass incomplete deep middle ...,1st & 10 | Q3 02:23 | WAS 44
2728,2022091807,1217,0.900144,pass,0,0.049999,pass,4.664171,2.40,22.0,4,1,2,05:30,LA,13,ATL,(5:30) M.Mariota pass short right to P.Hesse t...,4th & 1 | Q2 05:30 | LA 13
1655,2022091101,2027,0.899877,pass,0,-0.279998,pass,5.515262,3.94,22.0,1,9,2,00:40,CAR,9,CLE,(:40) M.Dunn reported in as eligible. J.Briss...,1st & 9 | Q2 00:40 | CAR 9
3303,2022091802,1125,0.898226,pass,0,0.370000,pass,6.235385,0.64,22.0,1,10,2,13:47,WAS,25,WAS,(13:47) C.Wentz pass incomplete short right to...,1st & 10 | Q2 13:47 | WAS 25
5400,2022092501,766,0.893133,pass,0,-0.420000,pass,4.106975,1.06,22.0,2,4,1,03:45,CHI,4,HOU,(3:45) D.Mills pass short left to J.Akins for ...,2nd & 4 | Q1 03:45 | CHI 4
1651,2022091101,1879,0.890365,pass,0,-0.960002,pass,6.034465,0.64,22.0,1,10,2,01:57,CLE,40,CLE,(1:57) J.Brissett pass incomplete short left t...,1st & 10 | Q2 01:57 | CLE 40
2760,2022091807,2293,0.887897,pass,0,-0.200001,pass,5.758993,8.13,22.0,2,7,3,06:33,LA,33,LA,(6:33) M.Stafford pass short middle intended f...,2nd & 7 | Q3 06:33 | LA 33
15147,2022110607,2333,0.884526,pass,0,0.130001,pass,5.398848,2.12,22.0,1,10,3,07:52,MIN,47,WAS,(7:52) T.Heinicke pass short right to T.McLaur...,1st & 10 | Q3 07:52 | MIN 47
13313,2022103009,1152,0.883484,pass,0,0.730000,pass,5.258377,0.90,22.0,1,10,2,08:02,WAS,34,WAS,(8:02) T.Heinicke pass short left to A.Gibson ...,1st & 10 | Q2 08:02 | WAS 34


In [6]:
#I want to compare the projected pass/run probability from situation (Cell 2) vs the coordinate-based model (Cell 3) for each play, and see how they align with the actual result. Can you help me create a summary table that shows these probabilities side by side along with the true label from plays.csv?

#To create a summary table that compares the projected pass/run probabilities from the situation model (Cell 2) and the coordinate-based model (Cell 3) alongside the actual result from `plays.csv`, you can follow these steps:
#1. Ensure you have the necessary data loaded from `plays.csv` and the predictions from both models.
#2. Create a DataFrame that consolidates the relevant information for each play, including the game
#Id, playId, projected probabilities from both models, and the actual result.
#Here's an example of how you can implement this in Python using pandas:

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

if 'predictions' not in globals() or not isinstance(predictions, pd.DataFrame):
    raise ValueError('Run Cell 3 first so predictions is available.')
if 'model' not in globals() or 'NUM_FEATS' not in globals() or 'CAT_FEATS' not in globals():
    raise ValueError('Run Cell 2 first so model, NUM_FEATS, and CAT_FEATS are available.')

plays_df = pd.read_csv(Path('.') / 'plays.csv')
coordinate_model_predictions = predictions[['gameId', 'playId', 'pred_pass_prob']].copy()
coordinate_model_predictions = coordinate_model_predictions.rename(columns={'pred_pass_prob': 'coordinate_pass_prob'})

keys = coordinate_model_predictions[['gameId', 'playId']].drop_duplicates()
base = keys.merge(
    plays_df[['gameId', 'playId', 'down', 'yardsToGo', 'quarter', 'absoluteYardlineNumber',
              'offenseFormation', 'receiverAlignment', 'gameClock',
              'preSnapHomeScore', 'preSnapVisitorScore', 'passResult', 'rushLocationType']],
    on=['gameId', 'playId'],
    how='left'
).copy()

def clock_to_seconds(val):
    try:
        m, s = str(val).split(':')
        return int(m) * 60 + int(s)
    except Exception:
        return np.nan

base['gameClockSeconds'] = base['gameClock'].apply(clock_to_seconds)
base['scoreDiff'] = pd.to_numeric(base['preSnapHomeScore'], errors='coerce').fillna(0) - pd.to_numeric(base['preSnapVisitorScore'], errors='coerce').fillna(0)

X_situation = base[NUM_FEATS + CAT_FEATS].copy()
for c in CAT_FEATS:
    X_situation[c] = X_situation[c].fillna('UNKNOWN').astype(str)

situation_model_predictions = keys.copy()
situation_model_predictions['situation_pass_prob'] = model.predict_proba(X_situation)[:, 1]

summary_df = keys.copy()
summary_df = summary_df.merge(situation_model_predictions, on=['gameId', 'playId'], how='left')
summary_df = summary_df.merge(coordinate_model_predictions, on=['gameId', 'playId'], how='left')
summary_df = summary_df.merge(base[['gameId', 'playId', 'passResult', 'rushLocationType']], on=['gameId', 'playId'], how='left')

def actual_label(pass_result, rush_loc):
    pr = str(pass_result).upper().strip()
    rl = str(rush_loc).strip().upper()
    if pr in {'C', 'I', 'IN', 'S'}:
        return 'PASS'
    if pr == 'R' or rl not in {'', 'NAN', 'NONE'}:
        return 'RUN'
    return 'UNKNOWN'

summary_df['actual_label'] = [actual_label(pr, rl) for pr, rl in zip(summary_df['passResult'], summary_df['rushLocationType'])]
summary_df['actual_is_pass'] = summary_df['actual_label'].map({'PASS': 1, 'RUN': 0})
summary_df['combined_pass_prob'] = 0.65 * summary_df['situation_pass_prob'] + 0.35 * summary_df['coordinate_pass_prob']
summary_df['situation_minus_coordinate'] = summary_df['situation_pass_prob'] - summary_df['coordinate_pass_prob']
summary_df['abs_gap'] = summary_df['situation_minus_coordinate'].abs()

summary_df['situation_call'] = np.where(summary_df['situation_pass_prob'] >= 0.5, 'PASS', 'RUN')
summary_df['visual_call'] = np.where(summary_df['coordinate_pass_prob'] >= 0.5, 'PASS', 'RUN')
summary_df['combined_call'] = np.where(summary_df['combined_pass_prob'] >= 0.5, 'PASS', 'RUN')
summary_df['situation_correct'] = (summary_df['situation_call'] == summary_df['actual_label']).astype(int)
summary_df['visual_correct'] = (summary_df['visual_call'] == summary_df['actual_label']).astype(int)
summary_df['combined_correct'] = (summary_df['combined_call'] == summary_df['actual_label']).astype(int)

summary_df = summary_df[['gameId', 'playId', 'situation_pass_prob', 'coordinate_pass_prob', 'combined_pass_prob',
                         'situation_minus_coordinate', 'abs_gap', 'actual_label', 'actual_is_pass',
                         'situation_call', 'visual_call', 'combined_call',
                         'situation_correct', 'visual_correct', 'combined_correct']]
summary_df = summary_df.sort_values(['gameId', 'playId']).reset_index(drop=True)

print(f'Summary rows: {len(summary_df):,}')
#display(summary_df.head(25))

eval_df = summary_df.loc[summary_df['actual_is_pass'].isin([0, 1])].copy()
if eval_df.empty:
    raise ValueError('No valid PASS/RUN labels found from plays.csv for evaluation.')

def evaluate_prob_model(df, prob_col, call_col):
    y_true = df['actual_is_pass'].astype(int).to_numpy()
    p = df[prob_col].clip(1e-6, 1 - 1e-6).to_numpy()
    y_hat = (df[call_col] == 'PASS').astype(int).to_numpy()
    return {
        'n_plays': int(len(df)),
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'roc_auc': float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_true, p)),
        'log_loss': float(log_loss(y_true, p)),
    }

eval_table = pd.DataFrame({
    'situational_xpass': evaluate_prob_model(eval_df, 'situation_pass_prob', 'situation_call'),
    'visual_xpass': evaluate_prob_model(eval_df, 'coordinate_pass_prob', 'visual_call'),
    'combined_expected': evaluate_prob_model(eval_df, 'combined_pass_prob', 'combined_call'),
}).T

print('\nOverall accuracy and probability quality metrics:')
display(eval_table)

best_acc_model = eval_table['accuracy'].idxmax()
best_brier_model = eval_table['brier'].idxmin()
best_auc_model = eval_table['roc_auc'].idxmax() if eval_table['roc_auc'].notna().any() else 'n/a'

print('\nConclusions:')
print(f"1) Best accuracy: {best_acc_model} ({eval_table.loc[best_acc_model, 'accuracy']:.3f})")
print(f"2) Best calibrated probabilities (lowest Brier): {best_brier_model} ({eval_table.loc[best_brier_model, 'brier']:.4f})")
if best_auc_model != 'n/a':
    print(f"3) Best separation (ROC AUC): {best_auc_model} ({eval_table.loc[best_auc_model, 'roc_auc']:.3f})")

# ── Weight Optimization ────────────────────────────────────────────────────────
# Find the blend weight (alpha) for situational model that minimises Brier score
# on all labeled plays from plays.csv.  This replaces the hardcoded 0.65/0.35.
# The result is stored as OPTIMAL_SITUATION_WEIGHT / OPTIMAL_VISUAL_WEIGHT

y_eval = eval_df['actual_is_pass'].astype(int).to_numpy()
p_sit  = eval_df['situation_pass_prob'].clip(1e-6, 1 - 1e-6).to_numpy()
p_vis  = eval_df['coordinate_pass_prob'].clip(1e-6, 1 - 1e-6).to_numpy()

def _brier_for_alpha(alpha):
    p_blend = np.clip(alpha * p_sit + (1 - alpha) * p_vis, 1e-6, 1 - 1e-6)
    return brier_score_loss(y_eval, p_blend)

opt_result = minimize_scalar(_brier_for_alpha, bounds=(0.0, 1.0), method='bounded')
OPTIMAL_SITUATION_WEIGHT = float(np.clip(opt_result.x, 0.0, 1.0))
OPTIMAL_VISUAL_WEIGHT    = 1.0 - OPTIMAL_SITUATION_WEIGHT

# Re-evaluate combined model with optimal weights for comparison
p_opt_blend = np.clip(OPTIMAL_SITUATION_WEIGHT * p_sit + OPTIMAL_VISUAL_WEIGHT * p_vis, 1e-6, 1 - 1e-6)
opt_call    = np.where(p_opt_blend >= 0.5, 'PASS', 'RUN')
opt_metrics = {
    'n_plays': int(len(eval_df)),
    'accuracy': float(accuracy_score(y_eval, (opt_call == 'PASS').astype(int))),
    'roc_auc': float(roc_auc_score(y_eval, p_opt_blend)) if len(np.unique(y_eval)) > 1 else np.nan,
    'brier': float(brier_score_loss(y_eval, p_opt_blend)),
    'log_loss': float(log_loss(y_eval, p_opt_blend)),
}

compare_table = eval_table.copy()
compare_table.loc['combined_optimized'] = opt_metrics

print(f'\n── Optimal blend weights (Brier-minimising) ─────────────────────────────')
print(f'  Situational xPass weight : {OPTIMAL_SITUATION_WEIGHT:.3f}  (was 0.650)')
print(f'  Visual xPass weight      : {OPTIMAL_VISUAL_WEIGHT:.3f}  (was 0.350)')
print(f'\n── Model comparison (fixed vs optimized blend) ──────────────────────────')
display(compare_table[['accuracy', 'roc_auc', 'brier', 'log_loss']])

brier_improvement = eval_table.loc['combined_expected', 'brier'] - opt_metrics['brier']
print(f'\nBrier improvement vs fixed 0.65/0.35 blend: {brier_improvement:+.5f}')


Summary rows: 16,124

Overall accuracy and probability quality metrics:


,n_plays,accuracy,roc_auc,brier,log_loss
situational_xpass,16124.0,0.741875,0.819058,0.170954,0.513339
visual_xpass,16124.0,0.427003,0.490846,0.572553,4.963990
combined_expected,16124.0,0.674212,0.822021,0.219568,0.628283



Conclusions:
1) Best accuracy: situational_xpass (0.742)
2) Best calibrated probabilities (lowest Brier): situational_xpass (0.1710)
3) Best separation (ROC AUC): combined_expected (0.822)

── Optimal blend weights (Brier-minimising) ─────────────────────────────
  Situational xPass weight : 0.997  (was 0.650)
  Visual xPass weight      : 0.003  (was 0.350)

── Model comparison (fixed vs optimized blend) ──────────────────────────


,accuracy,roc_auc,brier,log_loss
situational_xpass,0.741875,0.819058,0.170954,0.513339
visual_xpass,0.427003,0.490846,0.572553,4.963990
combined_expected,0.674212,0.822021,0.219568,0.628283
combined_optimized,0.742000,0.819089,0.170950,0.513422



Brier improvement vs fixed 0.65/0.35 blend: +0.04862
